# 04 · Build the cache (Kaggle, CPU)

**What this notebook does:** it reads every series of the 4,407 training studies **once**, converts them to
canonical volumes and writes `/kaggle/working/cache/<StudyInstanceUID>.npz`. The training notebook (05)
attaches this output and never touches the DICOMs.

**Why?** One study is ~150 DICOM files. Reading them every epoch makes training disk-bound (the main slowness of
the old pipeline). Preprocessing details: `03_dicom_preprocessing.ipynb`.

| setting | value | meaning |
|---|---|---|
| `n_slices` | 16 | evenly spaced slices per slot from the central 80% of the stack |
| `size` | 224 | pixels |
| `fov_mm` | 160 | field of view → 0.71 mm per pixel |
| slots | 6 | sagittal / coronal / axial × fluid-sensitive / T1 |

Expected output: ~12 GB (compressed npz), ~1 hour on 4 CPUs.

**Kaggle settings:** Accelerator = None (CPU), Internet = may be off,
Input = competition data + the `rsna-knee-src` dataset.

In [ ]:
# Add the project root (the folder that contains src/) to the Python path; works locally and on Kaggle.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk with pruning: never enters the competition DICOM folders (rglob took minutes)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ not found")

PROJECT_ROOT = add_project_root()
print("Project root:", PROJECT_ROOT)

## 0. Decoders for compressed DICOM

Some series are compressed with JPEG Lossless or JPEG 2000. pydicom cannot open them without extra packages,
and the Kaggle image does not ship them. Without the decoders those series would be skipped silently
(this happened in our first cache attempt). The wheels are inside `rsna-knee-src` (`third_party/wheels`),
so no internet is needed.

In [ ]:
from pathlib import Path
from src.config import IS_KAGGLE
from src.dicom import ensure_decoders

from src.config import find_input
WHEELS = [p for p in [find_input("third_party/wheels")] if p] if IS_KAGGLE else []
DECODERS = ensure_decoders(WHEELS)
print("DICOM decoders:", DECODERS)
if IS_KAGGLE:
    assert DECODERS["libjpeg"] and DECODERS["openjpeg"], "JPEG decoders could not be installed!"

In [ ]:
import time
import numpy as np
import pandas as pd

from src.config import DATA_DIR, EXPERIMENTS_DIR, IS_KAGGLE
from src.cache import CacheConfig, build_cache

SPLIT = "train"
N_WORKERS = 4
CFG = CacheConfig(n_slices=16, size=224, fov_mm=160.0)
OUT_DIR = EXPERIMENTS_DIR / "cache"

series = pd.read_csv(DATA_DIR / f"{SPLIT}_series.csv")
STUDIES = None  # None = hepsi
if not IS_KAGGLE:
    # Local dry run: only the test studies whose DICOMs are on disk
    SPLIT = "test"
    series = pd.read_csv(DATA_DIR / "test_series.csv")
    STUDIES = [s for s in series.StudyInstanceUID.unique() if (DATA_DIR / "test_series" / s).exists()]
    N_WORKERS = 1
print(f"Data: {DATA_DIR} | split: {SPLIT} | studies: {series.StudyInstanceUID.nunique() if STUDIES is None else len(STUDIES)}")

## 1. Build the cache
If the run is interrupted, simply run it again: existing `.npz` files are skipped.

In [ ]:
t0 = time.time()
meta = build_cache(series, DATA_DIR, SPLIT, OUT_DIR, CFG, n_workers=N_WORKERS, studies=STUDIES)
print(f"Bitti: {len(meta)} vaka, {(time.time() - t0) / 60:.1f} dk")

## 2. Quality control

Read these tables before training:
* **slots per study**: a study with 0 slots could not be read at all.
* **errors**: number of broken series / files.
* **side (laterality)**: from the tag or from the geometry; where both exist, their agreement shows how reliable
  the geometric method is.
* **plane mismatch**: the plane in the CSV differs from the plane derived from the geometry.

In [ ]:
meta = pd.read_csv(OUT_DIR / "meta.csv")
print("slots per study:\n", meta.n_slots.value_counts().sort_index().to_string())
print("\nstudies with errors:", meta.errors.fillna("").astype(bool).sum())
print("\nside source:\n", meta.side_source.value_counts(dropna=False).to_string())
print("\ntaraf:\n", meta.side.value_counts(dropna=False).to_string())
both = meta.dropna(subset=["side_tag", "side_geometry"])
if len(both):
    print(f"\netiket ile geometri uyumu: {(both.side_tag == both.side_geometry).mean():.3f} (n={len(both)})")
print("\nstudies with a plane mismatch:", meta.plane_mismatch.fillna("").astype(bool).sum())

In [ ]:
import os
size_gb = sum(f.stat().st_size for f in OUT_DIR.glob("*.npz")) / 1e9
print(f"Cache boyutu: {size_gb:.2f} GB, dosya: {len(list(OUT_DIR.glob('*.npz')))}")
errs = meta[meta.errors.fillna("").astype(bool)]
errs.head(20)

**Next:** attach this notebook's output as an input of `05_train_kaggle.ipynb`.